In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/telco-customer.csv")
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


### Handling missing values

In [3]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors='coerce')
df["TotalCharges"].isna().sum()

np.int64(11)

In [4]:
df["TotalCharges"] = df["TotalCharges"].fillna(
    df["TotalCharges"].median()
)
df["TotalCharges"].isna().sum()

np.int64(0)

### Feature Engineering 

In [5]:
df = df.drop(columns=["customerID"])

In [6]:
(df["tenure"] == 0).sum()

df["AvgMonthlyCharges"] = (df["TotalCharges"] / df["tenure"].replace(0, np.nan))
df["tenure"].isna().sum()
df[["tenure", "AvgMonthlyCharges"]].max()
df["AvgMonthlyCharges"] = df["AvgMonthlyCharges"].fillna(df["MonthlyCharges"])
df["AvgMonthlyCharges"].isna().sum()

np.int64(0)

In [7]:
df["InternetService"].unique()

<StringArray>
['DSL', 'Fiber optic', 'No']
Length: 3, dtype: str

In [8]:
df["HasInternetService"] = (
    df["InternetService"] != "No"
).astype(int)
(df["HasInternetService"] == 1).sum()

np.int64(5517)

In [9]:
services = [
    'PhoneService',
    'MultipleLines',
    'OnlineBackup',
    'DeviceProtection', 
    'TechSupport', 
    'StreamingTV',
    'StreamingMovies'
]
df["NumServices"] = (
    df[services].apply(lambda c: c.eq("Yes")).sum(axis=1)
)

In [10]:
df["ChargesPerServices"] = (
    df['MonthlyCharges'] / df["NumServices"]
)

In [11]:
df["ContractLevel"] = df["Contract"].map({
    "Month-to-month": 0,
    "One year" : 1,
    "Two year" : 2
})
df["ContractLevel"].unique()

array([0, 1, 2])

### Features created 
- ContractLevel - Month-to-month=0, One year=1, Two year=2 
- AvgMonthlyCharges = TotalCharges / tenure  
- ChargesPerService - MonthlyCharges / number of services
- HasInternetService - binary (Yes/No)


### Encoding Categorical values

In [12]:
binary_columns = [
    "Partner",
    "Dependents",
    "PhoneService",
    "PaperlessBilling",
]
for col in binary_columns:
    df[col] = df[col].map({'Yes': 1, 'No': 0 })
df.Dependents.isna().sum(
)

np.int64(0)

In [13]:
one_hot_cols = [
    "gender",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "PaymentMethod"
]

df = pd.get_dummies(df, columns=one_hot_cols,drop_first=True, dtype=int)


In [14]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["Churn"])
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y, 
    test_size=0.2,
    random_state=42,
    stratify=y,
)
print("Train:")
print(y_train.value_counts(normalize=True))

print("\nTest:")
print(y_test.value_counts(normalize=True))

Train:
Churn
No     0.734647
Yes    0.265353
Name: proportion, dtype: float64

Test:
Churn
No     0.734564
Yes    0.265436
Name: proportion, dtype: float64


In [15]:
train_indexes = X_train.index
test_indexes = X_test.index
pd.Series(train_indexes).to_csv("../data/processed/train_indexes.csv", index=False)
pd.Series(test_indexes).to_csv("../data/processed/test_indexes.csv", index=False)

In [17]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
numeric_cols = [
    'tenure',
    'MonthlyCharges',
    'TotalCharges',
]
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])